# Inference Benchmark Report

Generated from the per-run / centralized results DB (SPECIFICATIONS.md §15.1). Latency/SLO math is over the measurement phase only (§12.2).

In [ ]:
# Parameters (papermill-injected by tools/reports/render.py). Defaults self-run on the fixture.
db_path = None
run_id = None
out_dir = "."
sessions_per_user_per_hour = {}


In [ ]:
from pathlib import Path
import pandas as pd
from tools.reports import analysis, plots
plots.set_style()
out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
if db_path is None:                       # demo/self-test mode: build the fixture
    from tools.reports.fixtures import build_fixture_db
    db_path = str(out / "fixture_results.db")
    build_fixture_db(db_path)
    run_id = run_id or "fixtureA"
    sessions_per_user_per_hour = sessions_per_user_per_hour or {"chat-short-turns": 3600.0}
report = analysis.load_run(db_path, run_id)
mreq = analysis.measurement_requests(report)
print("loaded run", report.run_id, "—", len(mreq), "measurement-phase requests")


## Scenario & assumptions

In [ ]:
# Scenario & assumptions (§14.7) — read every chart below in this context.
display(pd.DataFrame(report.manifest.get("mix", [])))
for c in report.manifest.get("classes", []):
    print(f"# {c['name']} — {c.get('summary','')}")
    print("  modelled:", c.get("modelled"))
    print("  NOT modelled:", c.get("not_modelled"))   # must not be missed (§15.1)
    print("  assumptions:", c.get("assumptions"))
print("run assumptions:", report.manifest.get("run_assumptions"))


## Configuration

In [ ]:
# Configuration summary
display(pd.DataFrame([{"model": report.model, "backend": report.backend, **report.backend_config}]))


## System pre-checks

In [ ]:
# System pre-checks (§14.6) — warns/fails flagged at the top (§15.1)
sp = report.system_prechecks
if sp.empty:
    print("no system pre-checks recorded")
else:
    flagged = sp[sp["status"].isin(["warn", "fail"])]
    if not flagged.empty:
        print("⚠️  DEGRADED FOUNDATION — interpret all numbers below with care:")
        display(flagged[["metric", "measured", "expected", "status"]])
    display(sp[["metric", "measured", "expected", "tolerance_pct", "status"]])


## Model loading times

In [ ]:
# Model loading times (§10.2), per instance
cols = ["instance_id", "node", "model_load_total_s", "model_load_weights_s",
        "model_load_engine_init_s", "model_load_cuda_graph_capture_s", "model_load_inductor_compile_s"]
display(report.instances[[c for c in cols if c in report.instances.columns]])


## Latency vs λ

In [ ]:
# TTFT vs λ with the per-class SLO line (§15.1)
ttft_slo = next((s["threshold"] for s in report.slos if s["metric"] == "ttft_ms"), None)
fig = plots.latency_figure(report, "ttft_ms", slo_threshold=ttft_slo)
fig.savefig(out / "ttft.png", bbox_inches="tight"); fig


In [ ]:
# Inter-token latency (TPOT/ITL) vs λ
tpot_slo = next((s["threshold"] for s in report.slos if s["metric"] == "tpot_ms"), None)
fig = plots.latency_figure(report, "tpot_ms", slo_threshold=tpot_slo)
fig.savefig(out / "itl.png", bbox_inches="tight"); fig


In [ ]:
# Per-class breakdown (mixed runs, §11.4/§15.1): TTFT percentiles by scenario
display(analysis.latency_vs_lambda(mreq, "ttft_ms", by_scenario=True))
display(analysis.failure_rate_vs_lambda(mreq, by_scenario=True))


## SLO attainment & λ*

In [ ]:
# SLO attainment per λ and the derived λ* (§13.4)
att = analysis.evaluate_slos(report)
lam_star = analysis.lambda_star(report)
print("λ* =", lam_star)
display(att)


## Applied load (requests/s & concurrent sessions)

In [ ]:
# Applied load vs λ (§15.1): requests/s & concurrent sessions — the realized load the
# session-start rate λ hides. λ counts session starts; requests/s and concurrent sessions
# are what the backend actually carries (always report BOTH, per reports/STYLE.md).
display(analysis.applied_load_vs_lambda(report))
fig = plots.applied_load_figure([(report, plots.MODEL_COLOR, report.run_id)])
fig.savefig(out / "applied-load.png", bbox_inches="tight"); fig


## Supportable users

In [ ]:
# Supportable-users estimate at λ* (§15.1) — edit sessions_per_user_per_hour above
users = analysis.supportable_users(report, lam_star, sessions_per_user_per_hour)
display(users if not users.empty else "λ* undefined — extend the sweep toward lower rates")


## Response quality

In [ ]:
# Response quality (§13.5): Stage-A gate, Stage-B scores, capacity-vs-quality
q = analysis.quality_summary(report)
if q["quality_flagged"]:
    print("⚠️  QUALITY-FLAGGED: Stage-A gate failed under on_fail=continue (§15.1)")
display(q["compare"][["suite", "eval_concurrency", "score"]] if not q["compare"].empty
        else "no Stage-B quality rows")
runs = analysis.list_runs(db_path)
if len(runs) > 1:
    print("Capacity vs quality across deployment configs:")
    display(analysis.capacity_vs_quality(db_path, runs, sessions_per_user_per_hour))


## Hardware utilisation

In [ ]:
# Hardware utilisation vs λ — untapped headroom (§13.3/§15.1)
fig = plots.hardware_figure(report, ["gpu_sm_active_pct", "gpu_tensor_active_pct"])
if fig is not None:
    fig.savefig(out / "hardware.png", bbox_inches="tight")
fig if fig is not None else "no hardware telemetry recorded"


## Raw data

In [ ]:
# Raw per-rate-level table
raw = analysis.latency_vs_lambda(mreq, "ttft_ms").merge(
    analysis.failure_rate_vs_lambda(mreq), on="rate_lambda", how="outer", suffixes=("_ttft", "")
)
display(raw)
